# CoBasket: interactive development notebook

This notebook is a running workspace for exploring CoBasket as the package develops.

The current sections cover:

1. downloading and validating adjusted prices;
2. inspecting cache and provenance information;
3. calculating returns and correlations;
4. testing a proposed basket for cointegration;
5. constructing a spread and generating trading signals;
6. running the current train/test backtest.

The notebook is intended for learning and research. It does not provide investment advice.

## 0. Setup

Install the package in editable mode from the repository root so notebook imports reflect local code changes:

```bash
pip install -e ".[notebook]"
```

Adjusted prices include the effects of stock splits and dividends. This is similar to calibrating an instrument before comparing measurements made at different times: known discontinuities are removed from the historical series.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cobasket.backtest import backtest_single_basket
from cobasket.cointegration import build_spread, johansen_test
from cobasket.data import DataManager, ValidationError, validate_prices
from cobasket.signals import zscore_signal

pd.options.display.float_format = "{:.4f}".format

## 1. Download adjusted prices

A **ticker** is the short exchange symbol used to identify a traded asset. Here we use two large US oil companies as a simple example.

The returned table contains one adjusted closing price per trading day. The rows are aligned onto dates shared by both stocks.

In [ ]:
tickers = ["XOM", "CVX"]

manager = DataManager(
    cache_dir=Path("../price_cache"),
    cache_max_age_days=1.0,
)

prices = manager.prices(tickers, period="2y")
prices.tail()

### Inspect provenance

`last_metadata` records which symbols came from the local cache, which were downloaded, and which failed. This plays a role similar to provenance metadata in a scientific data product.

In [ ]:
manager.last_metadata

Running the same request again should normally produce cache hits rather than another network request.

In [ ]:
prices_cached = manager.prices(tickers, period="2y")
manager.last_metadata

## 2. Validate and visualize the data

Validation checks the data contract: a unique increasing datetime index, finite positive prices, numeric columns, and no missing values in the final aligned table.

In [ ]:
validate_prices(prices)
print(f"Rows: {len(prices)}")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
print(prices.describe())

In [ ]:
ax = prices.plot(figsize=(10, 4), title="Adjusted closing prices")
ax.set_ylabel("Adjusted price")
plt.show()

The absolute prices have different scales. Dividing each series by its first value gives a dimensionless relative trajectory, analogous to normalizing two experimental signals by their initial amplitudes.

In [ ]:
relative_prices = prices / prices.iloc[0]
ax = relative_prices.plot(figsize=(10, 4), title="Prices normalized to the first observation")
ax.set_ylabel("Relative price")
plt.show()

## 3. Returns and correlation

A daily **return** is the fractional price change,

$r_t = \frac{P_t-P_{t-1}}{P_{t-1}}$.

This is analogous to a discrete fractional derivative. Price levels often drift, while returns are usually closer to a stationary noisy process.

Correlation measures simultaneous linear co-movement. Correlation alone does **not** imply that a stable long-term relationship exists.

In [ ]:
returns = prices.pct_change().dropna()
returns.describe()

In [ ]:
returns.corr()

In [ ]:
ax = returns.plot(figsize=(10, 4), alpha=0.7, title="Daily fractional returns")
ax.set_ylabel("Return")
plt.show()

## 4. Test for cointegration

Two price series may each wander like random walks while a particular weighted combination remains bounded. That bounded combination is called a **cointegrating spread**.

A physics analogy is two drifting sensors whose absolute zero-points are unstable, but whose calibrated difference remains tied to an equilibrium value.

The Johansen test compares a test statistic with tabulated critical values. A statistic above the chosen critical value is evidence against the hypothesis of no cointegration. This is a statistical screening test, not proof that the relationship will persist.

In [ ]:
result = johansen_test(prices, verbose=True)

## 5. Construct the spread

The Johansen procedure returns a weight vector. The spread is

$S_t = \sum_i w_i P_{i,t}$.

The absolute scaling of the weights is arbitrary; multiplying all weights by the same constant changes only the numerical units of the spread.

In [ ]:
spread, weights = build_spread(prices, result)

pd.Series(weights, index=prices.columns, name="weight")

In [ ]:
ax = spread.plot(figsize=(10, 4), title="Cointegrating spread")
ax.set_ylabel("Spread")
plt.show()

## 6. Generate a mean-reversion signal

The rolling z-score measures displacement from a local mean in units of the local standard deviation:

$z_t = \frac{S_t-\mu_t}{\sigma_t}$.

This is similar to expressing a residual in units of its estimated noise scale.

The current signal rule is:

- `+1`: hold a long-spread position;
- `-1`: hold a short-spread position;
- `0`: hold no position.

A large positive z-score triggers a short-spread position because the strategy assumes the spread will move back toward its local mean. A large negative z-score triggers the opposite position.

In [ ]:
z, signal = zscore_signal(
    spread,
    window=30,
    entry_z=2.0,
    exit_z=0.5,
)

signal.value_counts().sort_index()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
spread.plot(ax=axes[0], title="Spread")
z.plot(ax=axes[1], title="Rolling z-score")
axes[1].axhline(2.0, linestyle="--")
axes[1].axhline(-2.0, linestyle="--")
axes[1].axhline(0.5, linestyle=":")
axes[1].axhline(-0.5, linestyle=":")
plt.tight_layout()
plt.show()

## 7. Run the current train/test backtest

The data are split into two chronological halves:

- the **estimation window** is used to infer the basket weights;
- the **trading window** is used to evaluate the frozen model.

This is analogous to fitting a model on a training dataset and evaluating it on held-out observations. It prevents the future trading interval from directly determining the fitted weights.

A **basis point** is one hundredth of a percentage point:

\[
1\ \mathrm{bp} = 0.01\% = 10^{-4}.
\]

`cost_bps=10` therefore applies a simplified cost of 0.1% when the position changes.

In [ ]:
backtest_result = backtest_single_basket(
    prices,
    tickers=tickers,
    cost_bps=10,
)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 11), sharex=True)
backtest_result["trading_prices"].plot(ax=axes[0], title="Trading-window prices")
backtest_result["trading_spread"].plot(ax=axes[1], title="Spread with frozen weights")
backtest_result["z"].plot(ax=axes[2], title="Z-score")
backtest_result["equity"].plot(ax=axes[3], title="Current equity curve")
plt.tight_layout()
plt.show()

## 8. Validation failure example

The validator should reject physically impossible or corrupted inputs instead of silently allowing them downstream.

In [ ]:
bad_prices = prices.copy()
bad_prices.iloc[0, 0] = -1.0

try:
    validate_prices(bad_prices)
except ValidationError as exc:
    print(type(exc).__name__, exc)

## 9. Exercises

Try changing one item at a time:

1. Replace the basket with another related pair, such as `KO` and `PEP`.
2. Increase the history from `2y` to `5y`.
3. Change the rolling window from 30 to 60 trading days.
4. Change the entry threshold from 2.0 to 1.5 or 2.5.
5. Compare the resulting number of trades and equity curves.

Do not interpret the most profitable historical setting as necessarily best. Searching many settings and retaining the winner is analogous to repeated hypothesis testing without correcting for the trials factor.